# 03 — JSON and CSV

## Learning Objectives

By the end of this notebook you will be able to:

- Load and save JSON files using the `json` module
- Read CSV files using `csv.reader`, `csv.DictReader`, and `pandas`
- Write CSV files using `csv.writer` and `csv.DictWriter`
- Use `pandas` basics: `pd.read_csv()`, `df.head()`, `df.shape`, `df.columns`, `df.groupby()`
- Work with the actual data files in this repository

## Setup

In [ ]:
import json
import csv
import os
import sys
from pathlib import Path

# Find the repo root by walking up until we find pyproject.toml
def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("Could not find repo root")

_REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(_REPO_ROOT))
from src.checks import check_equal, check_type, check_contains, check_length, check_keys

JSON_PATH = _REPO_ROOT / "data" / "synthetic" / "model_outputs.json"
CSV_PATH  = _REPO_ROOT / "data" / "synthetic" / "evaluation_results.csv"

os.makedirs("output", exist_ok=True)
print("Setup complete")
print("JSON_PATH exists:", JSON_PATH.exists())
print("CSV_PATH exists: ", CSV_PATH.exists())

## JSON — the language of APIs

JSON is how most AI APIs (OpenAI, Anthropic, etc.) return data — and how evaluation pipelines
store results. Python's built-in `json` module handles it natively.

JSON types map directly to Python types:

| JSON | Python |
|---|---|
| `object {}` | `dict` |
| `array []` | `list` |
| `string` | `str` |
| `number` | `int` or `float` |
| `true`/`false` | `True`/`False` |
| `null` | `None` |

In [ ]:
import json

# json.load(f) — read from an open file object
with open(JSON_PATH, "r", encoding="utf-8") as f:
    outputs = json.load(f)

print(f"Loaded {len(outputs)} records")
print("First record:")
print(json.dumps(outputs[0], indent=2))  # json.dumps() converts back to a formatted string

### The four `json` functions

The naming follows a consistent pattern: `load`/`dump` work with **files**, `loads`/`dumps` work with **strings** (the `s` stands for "string").

```
json.load(file)     → Python object     (read from file)
json.loads(string)  → Python object     (parse a string)
json.dump(obj, f)   → writes to file    (serialize to file)
json.dumps(obj)     → string            (serialize to string)
```

In [ ]:
# json.loads — parse a JSON string (common when receiving API responses)
raw_string = '{"model": "gpt-4", "score": 0.92, "flagged": false}'
parsed = json.loads(raw_string)
print(type(parsed))         # <class 'dict'>
print(parsed["model"])      # gpt-4
print(parsed["flagged"])    # False (Python bool, not JS false)

# json.dumps — serialize to a string (for sending or printing)
data = {"model": "claude-3", "scores": [0.91, 0.88], "passed": True}
as_string = json.dumps(data, indent=2)
print(as_string)

# json.dump — write to file
with open("output/sample.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)
print("Written to output/sample.json")

## CSV — tabular data

CSV files are spreadsheet-like: rows of comma-separated values, with an optional header row.
Python's `csv` module handles quoting, commas inside fields, and other edge cases correctly
(don't split on commas yourself — it breaks on quoted fields).

Two main readers:
- **`csv.reader`** — each row is a `list` of strings
- **`csv.DictReader`** — each row is a `dict` with column names as keys (usually what you want)

In [ ]:
import csv

# csv.reader — each row is a list
print("--- csv.reader ---")
with open(CSV_PATH, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)   # first row is the header
    print("Header:", header)
    first_row = next(reader)
    print("Row 1: ", first_row)

# csv.DictReader — each row is a dict keyed by header
print("\n--- csv.DictReader ---")
with open(CSV_PATH, "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        print(dict(row))
        if i >= 2:
            print("(... truncated)")
            break

### Writing CSV files

In [ ]:
# csv.DictWriter — write dicts to CSV
results = [
    {"model": "model-a", "task": "factual", "score": 0.92},
    {"model": "model-b", "task": "factual", "score": 0.74},
]
fieldnames = ["model", "task", "score"]

with open("output/sample_results.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()   # writes the header row
    writer.writerows(results)

print("Written. Contents:")
print(open("output/sample_results.csv").read())

## pandas — just enough to load and inspect data

`pandas` is the workhorse of data analysis in Python. A `DataFrame` is like a spreadsheet
or SQL table in memory. We'll cover pandas in depth in Module 04 — for now, just learn enough
to load a file and take a peek.

Key methods:
- `pd.read_csv(path)` — load a CSV file into a DataFrame
- `df.head(n)` — show the first n rows (default 5)
- `df.shape` — `(rows, columns)` tuple
- `df.columns` — column names
- `df.groupby(col)[col2].mean()` — group by a column and compute the mean of another

In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)          # (rows, columns)
print("Columns:", list(df.columns))
print()
print(df.head())                   # first 5 rows

In [ ]:
# groupby — compute per-model average score
per_model = df.groupby("model")["score"].mean()
print("Per-model average score:")
print(per_model)

## Your Turn — Exercises

### Exercise 1: Load the JSON file and count total records

Load `model_outputs.json` using `json.load()` and store:
- the full list of records in `model_outputs`
- the total count in `total_count`

In [ ]:
# YOUR CODE HERE
model_outputs = []

total_count = 0

In [ ]:
check_equal(total_count, 20, "total_count is 20")
check_type(model_outputs, list, "model_outputs is a list")
check_length(model_outputs, 20, "model_outputs has 20 items")

### Exercise 2: Find all flagged records

From `model_outputs`, find all records where `flagged == True`.
Store:
- the list of flagged record dicts in `flagged_records`
- a list of their `id` values in `flagged_ids`

In [ ]:
# YOUR CODE HERE
flagged_records = None
flagged_ids = None

In [ ]:
check_length(flagged_records, 7, "7 flagged records")
check_contains(flagged_ids, "out_003", "out_003 is flagged")
check_contains(flagged_ids, "out_005", "out_005 is flagged")

### Exercise 3: Compute average score using `csv.DictReader`

Load `evaluation_results.csv` using `csv.DictReader`.
Compute the average `score` across all rows. Store it in `avg_score`.

Reminder: values from `csv.DictReader` are **strings** — you'll need `float(row["score"])`.

In [ ]:
# YOUR CODE HERE
avg_score = None

In [ ]:
# avg_score should be 0.8225
check_equal(round(avg_score, 4), 0.8225, "average score is 0.8225")
check_type(avg_score, float, "avg_score is a float")

### Exercise 4: Per-model averages using pandas

Load `evaluation_results.csv` using `pd.read_csv()`. Use `df.groupby("model")["score"].mean()`
to compute per-model average scores. Store the result in `per_model_avg`.

Then extract the best model name (highest average score) into `best_model`.

In [ ]:
# YOUR CODE HERE
per_model_avg = None

best_model = None

In [ ]:
check_equal(best_model, "model-a-v2", "best model is model-a-v2")
check_equal(round(float(per_model_avg["model-a-v2"]), 4), 0.926, "model-a-v2 avg score")

## Why This Matters for AI Research Engineering

JSON is how AI APIs return model outputs — every call to the OpenAI or Anthropic API gives
you back a JSON object. When you run large-scale evaluations, results get stored as JSON files.

CSV is how benchmark scores and evaluation results get tabulated and shared — it's the
lingua franca of spreadsheets and data analysis.

In almost every research analysis you'll:
1. Load a JSON file of model outputs
2. Filter, count, and summarize them
3. Load a CSV of evaluation scores
4. Compute per-model or per-task statistics
5. Save results for reporting

You now have the tools to do all of that.

## Summary

| Task | Code |
|---|---|
| Load JSON from file | `json.load(f)` |
| Parse JSON string | `json.loads(s)` |
| Write JSON to file | `json.dump(obj, f, indent=2)` |
| Serialize to string | `json.dumps(obj, indent=2)` |
| Read CSV as lists | `csv.reader(f)` |
| Read CSV as dicts | `csv.DictReader(f)` |
| Write CSV | `csv.DictWriter(f, fieldnames=...)` |
| Load CSV to DataFrame | `pd.read_csv(path)` |
| Inspect DataFrame | `df.head()`, `df.shape`, `df.columns` |
| Group and aggregate | `df.groupby(col)[col2].mean()` |

**Next up:** `04_review_and_practice.ipynb` — putting it all together.